# Analysis of Electric Vehicle Adoption in the UK

## Notebook 1: Data Collection and Initial Inspection

This notebook presents the initial inspection of the datasets used in this study. The purpose is to understand the structure, content and quality of each dataset before any preprocessing is undertaken. The inspection provides an overview of the available variables and identifies any issues that may affect the later stages of the analysis.

The datasets examined in this notebook are:

1. Licensed plug-in vehicles by LSOA and quarter.
2. Public electric vehicle charging infrastructure.
3. LSOA 2021 to Local Authority District lookup.
4. LSOA population estimates.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Set the project folders

project_folder = Path.cwd().parent

raw_folder = project_folder / "Datasets" / "Raw"
processed_folder = project_folder / "Datasets" / "processed"
merged_folder = project_folder / "Datasets" / "merged"

print("Project folder:")
print(project_folder)

print()

print("Raw folder:")
print(raw_folder)

print()

print("Raw folder exists:", raw_folder.exists())

Project folder:
/Users/adityakumar/Desktop/EV_Dissertation

Raw folder:
/Users/adityakumar/Desktop/EV_Dissertation/Datasets/Raw

Raw folder exists: True


In [3]:
# Check the files in the Raw folder

raw_files = sorted(raw_folder.iterdir())

for file in raw_files:
    print(file.name)

.DS_Store
Lower_Layer_Super_Output_Area_(2021)_to_LAD_(April_2023)_Lookup_in_England_and_Wales.csv
df_VEH0145.csv
evci9001_2026-04_EV_charging_devices_UK.ods
sapelsoasyoa20222024.xlsx


In [4]:
# Save the path of each dataset

ev_file = raw_folder / "df_VEH0145.csv"

charging_file = (
    raw_folder
    / "evci9001_2026-04_EV_charging_devices_UK.ods"
)

lookup_file = (
    raw_folder
    / "Lower_Layer_Super_Output_Area_(2021)_to_LAD_(April_2023)_Lookup_in_England_and_Wales.csv"
)

population_file = raw_folder / "sapelsoasyoa20222024.xlsx"

files_to_check = {
    "EV data": ev_file,
    "Charging data": charging_file,
    "Geography lookup": lookup_file,
    "Population data": population_file
}

for dataset_name, file_path in files_to_check.items():
    print(f"{dataset_name}: {file_path.exists()}")

EV data: True
Charging data: True
Geography lookup: True
Population data: True


## 1. Licensed plug-in vehicle data

The main dataset contains quarterly counts of licensed plug-in vehicles at LSOA level. This section checks the size, columns and general structure of the data before any cleaning is carried out.

In [5]:
# Read the main EV dataset

ev_data = pd.read_csv(ev_file, low_memory=False)

print("EV dataset loaded successfully.")
print("Rows:", ev_data.shape[0])
print("Columns:", ev_data.shape[1])

EV dataset loaded successfully.
Rows: 285505
Columns: 62


In [6]:
ev_data.head()

,LSOA21CD,LSOA21NM,Fuel,Keepership,2026 Q1,2025 Q4,2025 Q3,2025 Q2,2025 Q1,2024 Q4,2024 Q3,2024 Q2,2024 Q1,2023 Q4,2023 Q3,2023 Q2,2023 Q1,2022 Q4,2022 Q3,2022 Q2,2022 Q1,2021 Q4,2021 Q3,2021 Q2,2021 Q1,2020 Q4,2020 Q3,2020 Q2,2020 Q1,2019 Q4,2019 Q3,2019 Q2,2019 Q1,2018 Q4,2018 Q3,2018 Q2,2018 Q1,2017 Q4,2017 Q3,2017 Q2,2017 Q1,2016 Q4,2016 Q3,2016 Q2,2016 Q1,2015 Q4,2015 Q3,2015 Q2,2015 Q1,2014 Q4,2014 Q3,2014 Q2,2014 Q1,2013 Q4,2013 Q3,2013 Q2,2013 Q1,2012 Q4,2012 Q3,2012 Q2,2012 Q1,2011 Q4
0,E01000001,City of London 001A,Battery electric,Company,13,10,10,10,9,10,8,9,8,8,7,6,8,8,7,5,5,[c],[c],[c],[c],[c],[c],[c],[c],[c],[c],[c],[c],0,0,[c],[c],[c],[c],[c],[c],[c],[c],0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,E01000002,City of London 001B,Battery electric,Company,12,14,11,11,12,9,9,9,9,9,8,18,18,17,17,13,7,7,7,[c],8,7,6,[c],[c],[c],[c],[c],[c],[c],[c],0,0,0,0,0,21,21,21,21,11,10,0,0,[c],[c],[c],[c],[c],[c],[c],[c],[c],[c],[c],[c],0,0
2,E01000005,City of London 001E,Battery electric,Company,18,18,22,21,18,18,19,18,17,81,80,83,77,76,10,7,6,6,[c],[c],0,0,0,0,0,0,0,0,0,0,[c],[c],[c],[c],[c],[c],[c],[c],[c],[c],[c],[c],[c],[c],[c],[c],[c],[c],0,0,0,0,0,0,0,0,0,0
3,E01000007,Barking and Dagenham 015A,Battery electric,Company,7,8,8,9,9,8,5,[c],[c],[c],[c],[c],[c],[c],[c],5,5,[c],[c],[c],[c],[c],0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,E01000008,Barking and Dagenham 015B,Battery electric,Company,134,129,125,124,117,99,91,77,53,45,52,34,25,20,19,19,15,11,5,[c],[c],[c],[c],[c],0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [7]:
# Display all column names

for column in ev_data.columns:
    print(column)

LSOA21CD
LSOA21NM
Fuel
Keepership
2026 Q1
2025 Q4
2025 Q3
2025 Q2
2025 Q1
2024 Q4
2024 Q3
2024 Q2
2024 Q1
2023 Q4
2023 Q3
2023 Q2
2023 Q1
2022 Q4
2022 Q3
2022 Q2
2022 Q1
2021 Q4
2021 Q3
2021 Q2
2021 Q1
2020 Q4
2020 Q3
2020 Q2
2020 Q1
2019 Q4
2019 Q3
2019 Q2
2019 Q1
2018 Q4
2018 Q3
2018 Q2
2018 Q1
2017 Q4
2017 Q3
2017 Q2
2017 Q1
2016 Q4
2016 Q3
2016 Q2
2016 Q1
2015 Q4
2015 Q3
2015 Q2
2015 Q1
2014 Q4
2014 Q3
2014 Q2
2014 Q1
2013 Q4
2013 Q3
2013 Q2
2013 Q1
2012 Q4
2012 Q3
2012 Q2
2012 Q1
2011 Q4


In [8]:
# Identify the quarterly columns

quarter_columns = [
    column
    for column in ev_data.columns
    if "Q" in str(column)
]

print("Number of quarter columns:", len(quarter_columns))
print("First quarter:", quarter_columns[-1])
print("Latest quarter:", quarter_columns[0])

Number of quarter columns: 58
First quarter: 2011 Q4
Latest quarter: 2026 Q1


In [9]:
# Count the different fuel types

ev_data["Fuel"].value_counts(dropna=False)

Fuel
Total                               110156
Battery electric                    100740
Plug-in hybrid electric (petrol)     73153
Range extended electric               1097
Plug-in hybrid electric (diesel)       359
Name: count, dtype: int64

In [10]:
# Count the different keepership categories

ev_data["Keepership"].value_counts(dropna=False)

Keepership
Total       119987
Private     111794
Company      53719
Disposal         5
Name: count, dtype: int64

## EV Dataset Overview

The EV registration dataset was examined to understand its overall structure and assess its suitability for the analysis. The inspection focused on the dataset dimensions, variable types, completeness and data quality before any preprocessing was undertaken. This provides an understanding of the available information and identifies any issues that may require attention during the data cleaning stage.

In [11]:
# Display the structure of the EV dataset

ev_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 285505 entries, 0 to 285504
Data columns (total 62 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   LSOA21CD    285505 non-null  object
 1   LSOA21NM    285505 non-null  object
 2   Fuel        285505 non-null  object
 3   Keepership  285505 non-null  object
 4   2026 Q1     285505 non-null  object
 5   2025 Q4     285505 non-null  object
 6   2025 Q3     285505 non-null  object
 7   2025 Q2     285505 non-null  object
 8   2025 Q1     285505 non-null  object
 9   2024 Q4     285505 non-null  object
 10  2024 Q3     285505 non-null  object
 11  2024 Q2     285505 non-null  object
 12  2024 Q1     285505 non-null  object
 13  2023 Q4     285505 non-null  object
 14  2023 Q3     285505 non-null  object
 15  2023 Q2     285505 non-null  object
 16  2023 Q1     285505 non-null  object
 17  2022 Q4     285505 non-null  object
 18  2022 Q3     285505 non-null  object
 19  2022 Q2     285505 non-

In [12]:
# Check for missing values

missing_values = (
    ev_data.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_values[missing_values > 0]

Series([], dtype: int64)

In [13]:
# Check for duplicate records

duplicate_rows = ev_data.duplicated().sum()

print(f"Duplicate rows: {duplicate_rows}")

Duplicate rows: 0


In [14]:
# Check the memory used by the dataset

memory_usage = ev_data.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"Dataset memory usage: {memory_usage:.2f} MB")

Dataset memory usage: 867.77 MB


In [15]:
# Generate summary statistics for the quarterly columns

ev_data[quarter_columns].describe().T

,count,unique,top,freq
2026 Q1,285505,852,7,13252
2025 Q4,285505,829,6,15770
2025 Q3,285505,828,5,19451
2025 Q2,285505,811,5,19812
2025 Q1,285505,783,[c],27310
2024 Q4,285505,758,[c],37909
2024 Q3,285505,732,[c],47726
2024 Q2,285505,721,[c],59181
2024 Q1,285505,724,[c],69335
2023 Q4,285505,699,[c],78288


## Geography Lookup Dataset

The geography lookup dataset provides the relationship between LSOA 2021 codes and Local Authority Districts (LADs). This dataset is required to aggregate the EV registration data from LSOA level to local authority level before it is combined with the charging infrastructure and population datasets.

In [16]:
# Read the geography lookup dataset

lookup_data = pd.read_csv(lookup_file)

print("Geography lookup loaded successfully.")
print("Rows:", lookup_data.shape[0])
print("Columns:", lookup_data.shape[1])

Geography lookup loaded successfully.
Rows: 35672
Columns: 7


In [17]:
# Display the first five rows

lookup_data.head()

,LSOA21CD,LSOA21NM,LSOA21NMW,LAD23CD,LAD23NM,LAD23NMW,ObjectId
0,E01011949,Hartlepool 009A,NaN,E06000001,Hartlepool,NaN,1
1,E01011950,Hartlepool 008A,NaN,E06000001,Hartlepool,NaN,2
2,E01011951,Hartlepool 007A,NaN,E06000001,Hartlepool,NaN,3
3,E01011952,Hartlepool 002A,NaN,E06000001,Hartlepool,NaN,4
4,E01011953,Hartlepool 002B,NaN,E06000001,Hartlepool,NaN,5


In [18]:
# Display the dataset structure

lookup_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35672 entries, 0 to 35671
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   LSOA21CD   35672 non-null  object
 1   LSOA21NM   35672 non-null  object
 2   LSOA21NMW  1917 non-null   object
 3   LAD23CD    35672 non-null  object
 4   LAD23NM    35672 non-null  object
 5   LAD23NMW   1917 non-null   object
 6   ObjectId   35672 non-null  int64 
dtypes: int64(1), object(6)
memory usage: 1.9+ MB


In [19]:
# Display the column names

lookup_data.columns.tolist()

['LSOA21CD',
 'LSOA21NM',
 'LSOA21NMW',
 'LAD23CD',
 'LAD23NM',
 'LAD23NMW',
 'ObjectId']

In [20]:
# Check for missing values

lookup_missing = (
    lookup_data.isna()
    .sum()
    .sort_values(ascending=False)
)

lookup_missing[lookup_missing > 0]

LSOA21NMW    33755
LAD23NMW     33755
dtype: int64

In [21]:
# Count duplicate rows

lookup_duplicates = lookup_data.duplicated().sum()

print(f"Duplicate rows: {lookup_duplicates}")

Duplicate rows: 0


In [22]:
# Check the number of unique LSOA codes

print("Unique LSOA codes:", lookup_data["LSOA21CD"].nunique())
print("Total rows:", len(lookup_data))

Unique LSOA codes: 35672
Total rows: 35672


### Interpretation

The geography lookup dataset was successfully loaded and inspected. The dataset contains 35,672 unique LSOA 2021 records, with each row representing a single geographical area. No duplicate records were identified, confirming that each LSOA code appears only once.

Missing values were found only in the Welsh-language name fields (`LSOA21NMW` and `LAD23NMW`). These missing values are expected because English areas do not have Welsh-language names. Therefore, these values do not affect the quality of the dataset or its suitability for linking the EV registration, charging infrastructure and population datasets.

## Public EV Charging Infrastructure Dataset

The public charging infrastructure dataset contains official statistics on charging devices across the United Kingdom. This dataset will be used to examine the growth and geographical distribution of charging infrastructure and to compare infrastructure development with EV adoption trends.

In [23]:
# Open the charging infrastructure workbook

charging_workbook = pd.ExcelFile(
    charging_file,
    engine="odf"
)

charging_workbook.sheet_names

['Cover',
 'Contents',
 'Notes',
 '1a',
 '1b',
 '2a',
 '2b',
 '3',
 '4',
 '5',
 '6a',
 '6b',
 '6c',
 '6d',
 '7a',
 '7b']

In [25]:
# Load the charging infrastructure dataset

charging_data = pd.read_excel(
    charging_file,
    sheet_name="1a",
    engine="odf",
    header=2
)

print("Rows:", charging_data.shape[0])
print("Columns:", charging_data.shape[1])

charging_data.head()

Rows: 433
Columns: 29


,Local authority / region code,Local authority / region name,Oct-19,Jan-20,Apr-20,Jul-20,Oct-20,Jan-21,Apr-21,Jul-21,Oct-21,Jan-22,Apr-22,Jul-22,Oct-22,Jan-23,Apr-23,Jul-23,Oct-23,Jan-24,Apr-24,Jul-24,Oct-24 [Note 14],Jan-25,Apr-25,Jul-25,Oct-25,Jan-26,Apr-26
0,K02000001,United Kingdom,15116,16505,17947,18265,19487,20775,22790,24374,25927,28375,30290,32011,34637,37055,40150,44020,49220,53677,59670,64632,70042,73334,76507,82002,86021,87796,92141
1,K03000001,Great Britain,14821,16210,17642,17953,19169,20455,22463,24044,25595,28030,29942,31683,34295,36689,39761,43587,48789,53212,59125,64014,69403,72654,75835,81318,85283,87069,91414
2,E92000001,England,12549,13719,14979,15395,16456,17459,19261,20563,21925,24159,25884,27502,29774,31466,34203,37717,42489,46374,51506,55631,60364,63389,65877,70672,74115,75818,79198
3,E12000001,North East,738,752,786,812,849,820,854,887,916,975,1011,1155,1142,1253,1392,1657,1536,1596,1942,1919,2295,2583,2553,2703,2698,2734,2950
4,E06000047,County Durham,92,96,102,105,106,110,121,116,124,128,149,174,206,229,240,259,268,291,309,313,359,400,462,482,482,489,523


In [26]:
# Display the dataset structure

charging_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 433 entries, 0 to 432
Data columns (total 29 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   Local authority / region code  433 non-null    object
 1   Local authority / region name  433 non-null    object
 2   Oct-19                         433 non-null    object
 3   Jan-20                         433 non-null    object
 4   Apr-20                         433 non-null    object
 5   Jul-20                         433 non-null    object
 6   Oct-20                         433 non-null    object
 7   Jan-21                         433 non-null    object
 8   Apr-21                         433 non-null    object
 9   Jul-21                         433 non-null    object
 10  Oct-21                         433 non-null    object
 11  Jan-22                         433 non-null    object
 12  Apr-22                         433 non-null    object
 13  Jul-2

In [27]:
# Display the column names

charging_data.columns.tolist()

['Local authority / region code',
 'Local authority / region name',
 'Oct-19',
 'Jan-20',
 'Apr-20',
 'Jul-20',
 'Oct-20',
 'Jan-21',
 'Apr-21',
 'Jul-21',
 'Oct-21',
 'Jan-22',
 'Apr-22',
 'Jul-22',
 'Oct-22',
 'Jan-23',
 'Apr-23',
 'Jul-23',
 'Oct-23',
 'Jan-24',
 'Apr-24',
 'Jul-24',
 'Oct-24 [Note 14]',
 'Jan-25',
 'Apr-25',
 'Jul-25',
 'Oct-25',
 'Jan-26',
 'Apr-26']

In [28]:
# Check for missing values

charging_missing = (
    charging_data.isna()
    .sum()
    .sort_values(ascending=False)
)

charging_missing[charging_missing > 0]

Series([], dtype: int64)

In [29]:
# Count duplicate rows

charging_duplicates = charging_data.duplicated().sum()

print(f"Duplicate rows: {charging_duplicates}")

Duplicate rows: 0


### Interpretation

The charging infrastructure dataset was successfully loaded and inspected. The dataset contains 433 geographical records and 29 variables representing quarterly counts of publicly available charging devices from October 2019 to April 2026.

No missing values or duplicate records were identified during the initial inspection. Although all quarterly columns were imported as object variables, this is a common outcome when reading spreadsheet files and will be corrected during preprocessing. One column name includes a reference to a spreadsheet note (`Oct-24 [Note 14]`), which will also be standardised during the data cleaning stage.

## Population Dataset

The population dataset contains official mid-year population estimates for Lower Layer Super Output Areas (LSOAs) in England and Wales. These estimates will be used to calculate population-based indicators and support comparisons between EV adoption, charging infrastructure and regional population characteristics.

In [30]:
# Open the population workbook

population_workbook = pd.ExcelFile(population_file)

population_workbook.sheet_names

['Cover sheet',
 'Contents',
 'Notes',
 'Related publications',
 'Mid-2022 LSOA 2021',
 'Mid-2023 LSOA 2021',
 'Mid-2024 LSOA 2021']

In [34]:
# Load the population dataset

population_data = pd.read_excel(
    population_file,
    sheet_name="Mid-2024 LSOA 2021",
    header=3
)

print("Population dataset loaded successfully.")
print("Rows:", population_data.shape[0])
print("Columns:", population_data.shape[1])

population_data.head()

Population dataset loaded successfully.
Rows: 35672
Columns: 187


,LAD 2023 Code,LAD 2023 Name,LSOA 2021 Code,LSOA 2021 Name,Total,F0,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15,F16,F17,F18,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28,F29,F30,F31,F32,F33,F34,F35,F36,F37,F38,F39,F40,F41,F42,F43,F44,...,M41,M42,M43,M44,M45,M46,M47,M48,M49,M50,M51,M52,M53,M54,M55,M56,M57,M58,M59,M60,M61,M62,M63,M64,M65,M66,M67,M68,M69,M70,M71,M72,M73,M74,M75,M76,M77,M78,M79,M80,M81,M82,M83,M84,M85,M86,M87,M88,M89,M90
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1898,6,7,16,3,11,21,8,9,11,14,9,9,8,15,9,14,17,16,16,8,8,10,10,7,9,11,9,7,17,12,11,11,8,11,15,18,10,14,13,16,13,11,19,16,8,...,11,5,17,11,11,8,13,7,12,8,14,9,7,13,15,6,20,14,14,15,20,7,11,8,9,10,7,10,9,12,12,6,8,4,2,7,2,4,4,3,3,3,4,5,1,1,2,2,2,3
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1247,11,10,7,14,13,15,3,10,14,12,12,7,5,11,8,7,4,5,9,6,2,3,3,2,3,7,7,8,7,10,7,7,7,10,12,11,7,18,22,16,6,2,9,7,7,...,6,7,10,6,12,10,11,7,8,6,6,8,8,6,12,6,11,10,9,6,13,12,13,4,9,5,13,9,5,5,8,7,3,3,1,6,1,0,0,2,0,2,1,0,0,1,0,0,0,0
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1393,5,11,9,9,12,12,10,8,8,8,6,3,5,7,8,4,7,3,9,5,8,7,10,19,4,18,10,11,12,18,12,7,15,11,11,11,15,12,14,10,15,15,11,6,3,...,9,10,8,8,8,6,8,10,12,9,17,10,7,13,8,5,10,3,16,15,15,12,10,8,7,8,8,5,6,5,2,2,3,3,8,3,1,4,2,2,1,0,0,1,3,2,0,1,0,1
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1669,10,12,12,10,17,13,13,8,9,12,11,9,12,10,12,13,10,11,15,9,12,10,3,7,9,12,9,12,9,12,12,11,9,12,17,11,22,17,13,14,9,6,6,4,9,...,8,5,5,15,5,2,8,8,7,11,3,10,8,8,10,14,9,11,19,11,22,11,8,7,8,6,5,6,5,5,3,8,7,7,5,8,8,9,5,5,0,3,2,1,3,3,3,2,4,5
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,2303,15,15,7,20,22,18,14,32,15,21,14,15,11,11,14,12,21,26,21,22,17,18,12,14,19,27,14,18,12,22,11,21,24,18,22,17,14,17,16,18,12,16,11,12,13,...,18,11,15,17,5,10,9,14,13,14,5,9,13,10,7,15,9,11,21,15,9,11,14,15,11,10,12,13,7,11,7,9,10,5,4,5,5,5,4,2,3,3,4,3,3,1,0,1,0,2


In [35]:
# Display the dataset structure

population_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35672 entries, 0 to 35671
Columns: 187 entries, LAD 2023 Code to M90
dtypes: int64(183), object(4)
memory usage: 50.9+ MB


In [36]:
# Display the column names

population_data.columns.tolist()

['LAD 2023 Code',
 'LAD 2023 Name',
 'LSOA 2021 Code',
 'LSOA 2021 Name',
 'Total',
 'F0',
 'F1',
 'F2',
 'F3',
 'F4',
 'F5',
 'F6',
 'F7',
 'F8',
 'F9',
 'F10',
 'F11',
 'F12',
 'F13',
 'F14',
 'F15',
 'F16',
 'F17',
 'F18',
 'F19',
 'F20',
 'F21',
 'F22',
 'F23',
 'F24',
 'F25',
 'F26',
 'F27',
 'F28',
 'F29',
 'F30',
 'F31',
 'F32',
 'F33',
 'F34',
 'F35',
 'F36',
 'F37',
 'F38',
 'F39',
 'F40',
 'F41',
 'F42',
 'F43',
 'F44',
 'F45',
 'F46',
 'F47',
 'F48',
 'F49',
 'F50',
 'F51',
 'F52',
 'F53',
 'F54',
 'F55',
 'F56',
 'F57',
 'F58',
 'F59',
 'F60',
 'F61',
 'F62',
 'F63',
 'F64',
 'F65',
 'F66',
 'F67',
 'F68',
 'F69',
 'F70',
 'F71',
 'F72',
 'F73',
 'F74',
 'F75',
 'F76',
 'F77',
 'F78',
 'F79',
 'F80',
 'F81',
 'F82',
 'F83',
 'F84',
 'F85',
 'F86',
 'F87',
 'F88',
 'F89',
 'F90',
 'M0',
 'M1',
 'M2',
 'M3',
 'M4',
 'M5',
 'M6',
 'M7',
 'M8',
 'M9',
 'M10',
 'M11',
 'M12',
 'M13',
 'M14',
 'M15',
 'M16',
 'M17',
 'M18',
 'M19',
 'M20',
 'M21',
 'M22',
 'M23',
 'M24',
 'M25',


In [37]:
# Check for missing values

population_missing = (
    population_data.isna()
    .sum()
    .sort_values(ascending=False)
)

population_missing[population_missing > 0]

Series([], dtype: int64)

In [38]:
# Count duplicate rows

population_duplicates = population_data.duplicated().sum()

print(f"Duplicate rows: {population_duplicates}")

Duplicate rows: 0


In [39]:
# Check the number of unique LSOA codes

print("Unique LSOA codes:", population_data["LSOA 2021 Code"].nunique())
print("Total rows:", len(population_data))

Unique LSOA codes: 35672
Total rows: 35672


### Interpretation

The population dataset was successfully loaded using the correct worksheet header. It contains 35,672 unique LSOA 2021 records and 187 variables. No missing values or duplicate records were identified, confirming that the dataset provides one complete record for each LSOA.

The dataset includes geographical identifiers, total population and detailed population estimates by sex and single year of age. The main analysis will use the LSOA and Local Authority District identifiers together with the total population field. The age- and sex-specific variables are retained in the raw data but are not required for the core objectives of this study.

## Summary

The four datasets required for the study were successfully loaded and inspected. The EV registration dataset provides quarterly plug-in vehicle counts from 2011 Q4 to 2026 Q1 at LSOA level. The geography lookup provides the relationship between LSOA codes and Local Authority Districts, while the charging infrastructure dataset contains quarterly public charging device counts from October 2019 to April 2026. The population dataset provides mid-2024 population estimates for each LSOA.

The initial inspection identified no duplicated records in any of the supporting datasets. The geography lookup contains expected missing values in Welsh-language name fields, while the EV dataset includes disclosure-controlled values and quarterly columns stored as text. These issues will be addressed in the data cleaning stage.

The next notebook will clean and standardise the datasets, reshape the EV and charging data into long format, and prepare the variables required for integration.